# Distinguished PC member analysis

Runs **entirely locally** using `../output/bak/pc_metrics.csv`, produced by `pc_metrics.py`.
Run cells top-to-bottom, then rerun any query independently. Requires `pandas` and `matplotlib` in your Jupyter environment (`pip install pandas matplotlib`).

**Important:** Wordcounts below refer to the final reviews. Comment totals include both ordinary and `extra` comments where indicated; all ranking tables label the metric. For early submissions, first/second mean each reviewer's first/second submission **within a given calendar round**.

## 1. Load CSV into a queryable DataFrame

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

csv_path = Path("../output/pc_metrics.csv")  # relative to the notebook working directory

df = pd.read_csv(csv_path, encoding="utf-8-sig")

# Lists/dictionaries were serialized to JSON inside cells of the CSV.
json_columns = [
    "bids", "reviews", "assignments", "time-to-review", "delays",
    "review_length", "hotcrp_length", "review_details", "discussion_initiator",
    "comments", "comments_length", "extra_comments", "extra_comments_length",
]
for col in json_columns:
    default = {} if col in {"comments", "comments_length", "extra_comments", "extra_comments_length"} else []
    df[col] = df[col].map(lambda x: json.loads(x) if isinstance(x, str) and x.strip() else default.copy())

df["reviewer"] = df["given_name"].fillna("") + " " + df["family_name"].fillna("")
df["topics_set"] = df["topics?"].astype(str).str.strip().str.casefold().map({"true": True, "false": False})

def all_comment_lengths(row):
    # Two nonoverlapping categories: comments on reviewed papers + extra comments.
    return [int(length)
            for column in ("comments_length", "extra_comments_length")
            for lengths in row[column].values()
            for length in lengths]

df["all_comment_lengths"] = df.apply(all_comment_lengths, axis=1)
df["all_final_review_lengths"] = df["review_length"].map(
    lambda words: [int(v) for v in words if v is not None]
)

print(f"Loaded {len(df)} PC members; {df['reviews_totCount'].sum()} final reviews")
display(df[["reviewer", "email", "topics?", "bids_totCount", "reviews_totCount",
            "all_comments_totCount", "discussion_initiator_totCount"]].head())

### Example query (modify and rerun)

In [ ]:
# Example: members with at least 3 submitted reviews and at least 5 comments.
display(
    df.query("reviews_totCount >= 3 and all_comments_totCount >= 5")
      [["reviewer", "email", "reviews_totCount", "all_comments_totCount"]]
      .sort_values("all_comments_totCount", ascending=False)
)

## 2. Distributions of comments and final review lengths

In [ ]:
# Four separate histograms: two per reviewer and two per individual contribution.
comment_words = [length for values in df["all_comment_lengths"] for length in values]
review_words = [length for values in df["all_final_review_lengths"] for length in values]

charts = [
    (df["all_comments_totCount"], "Comments per PC member", "Number of comments", 25),
    (df["all_comments_totalWords"], "Cumulative comment length per PC member", "Total comment words", 25),
    (comment_words, "Length of individual comments", "Words per comment", 35),
    (review_words, "Length of individual FINAL reviews", "Words per review", 35),
]
for values, title, xlabel, bins in charts:
    plt.figure(figsize=(8, 4))
    plt.hist(values, bins=bins, edgecolor="black")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("PC members" if "per PC member" in title else "Contributions")
    plt.tight_layout()
    plt.show()

## 3. Reviewers who initiated discussions

In [ ]:
initiators = (
    df.loc[df["discussion_initiator_totCount"] > 0,
           ["reviewer", "email", "discussion_initiator_totCount", "discussion_initiator"]]
      .sort_values(["discussion_initiator_totCount", "reviewer"], ascending=[False, True])
)
display(initiators)

# One record per earned initiation credit (a paper can appear once per round).
initiation_events = pd.DataFrame([
    {"reviewer": row.reviewer, "email": row.email, **event}
    for row in df.itertuples(index=False)
    for event in row.discussion_initiator
])
if not initiation_events.empty:
    display(initiation_events.sort_values(["round", "paper_id", "reviewer"]))

## 4. Topic declarations and bids

In [ ]:
columns = ["reviewer", "email", "topics?", "bids_totCount", "bids"]
both = df.loc[df["topics_set"].eq(True) & df["bids_totCount"].gt(0), columns]
neither = df.loc[df["topics_set"].eq(False) & df["bids_totCount"].eq(0), columns]

print(f"Set ALL topics and made at least one bid: {len(both)}")
display(both.sort_values("bids_totCount", ascending=False))
print(f"Did neither (topics explicitly incomplete and zero bids): {len(neither)}")
display(neither.sort_values("reviewer"))
# N/A topics are deliberately not included in 'neither'.

## 5. Earliest first and second reviews, measured from round start

In [ ]:
# Convert each review_details element into one row.
review_events = pd.DataFrame([
    {"reviewer": row.reviewer, "email": row.email,
     "paper_id": rec["paper_id"], "round": rec.get("round"),
     "submitted_at": rec.get("first_submitted_at"),
     "round_start": rec.get("round_start")}
    for row in df.itertuples(index=False)
    for rec in row.review_details
])

review_events["submitted_at"] = pd.to_datetime(review_events["submitted_at"], utc=True)
review_events["round_start"] = pd.to_datetime(review_events["round_start"], utc=True)
review_events["hours_from_round_start"] = (
    (review_events["submitted_at"] - review_events["round_start"]).dt.total_seconds() / 3600
)
# Only reviews submitted after their round started, with a known start timestamp.
review_events = review_events.dropna(subset=["round", "submitted_at", "round_start"])
review_events = review_events.loc[review_events["hours_from_round_start"] >= 0].copy()
review_events = review_events.sort_values(["round", "email", "submitted_at", "paper_id"])
review_events["review_number_in_round"] = (
    review_events.groupby(["round", "email"]).cumcount() + 1
)

for rnd in sorted(review_events["round"].unique()):
    for kth in (1, 2):
        print(f"Round {int(rnd)} — earliest {kth}{'st' if kth == 1 else 'nd'} review")
        leaders = review_events.loc[
            (review_events["round"] == rnd) & (review_events["review_number_in_round"] == kth)
        ].nsmallest(7, "hours_from_round_start")
        display(leaders[["reviewer", "email", "paper_id", "submitted_at", "hours_from_round_start"]])

## 6. Reviewers never late (across every round)

In [ ]:
# Exclude zero-review PC members, and require a known delay for every submitted review.
on_time = df.loc[
    df["reviews_totCount"].gt(0)
    & df["delays_knownCount"].eq(df["reviews_totCount"])
    & df["delays_lateCount"].eq(0),
    ["reviewer", "email", "reviews_totCount", "delays", "delays_lateCount"]
].sort_values(["reviews_totCount", "reviewer"], ascending=[False, True])

print(f"Reviewers with no late submissions across any round: {len(on_time)}")
display(on_time)

## 7. Top: total review words, total comment words, and comment count

In [ ]:
rankings = [
    ("Total words in final reviews", "review_length_totalWords"),
    ("Total words in ALL PC comments (ordinary + extra)", "all_comments_totalWords"),
    ("Number of ALL PC comments (ordinary + extra)", "all_comments_totCount"),
]
for title, metric in rankings:
    print(f"TOP 5% — {title}")
    display(df.nlargest(10, metric)[["reviewer", "email", "reviews_totCount", "all_comments_totCount", metric]])

b## Notes on interpretation

- **Final review length:** `review_length` / `review_length_totalWords`, not HotCRP's `hotcrp_length` (which is tracked separately).
- **Comments:** `all_comments_*` covers both regular and extra categories. `comments_*` alone covers comments posted while assigned on papers the PC member reviewed. Comments marked as ignored after unassignment in `pc_metrics.py` are not included in either category.
- **On time:** negative `delays` mean early submission; `0` means on-time or excused by the late-assignment rule. Unknown delays disqualify a reviewer from the strict "never late" table.
- **Fastest reviews:** Rankings compare the first and second formally submitted review of each person **within each round**, measured in elapsed hours from that round's start. This measures promptness relative to the calendar start, not to assignment time.
- **Topics:** `N/A` means no topic assessment is available, and must not be mistaken for a member who failed to fill it out.

In [ ]:
df

In [ ]:

import json
import pandas as pd
from IPython.display import display

TOP_N = 10
FIREFIGHTERS = set()  # Optional: {"reviewer@university.edu"}
BID_THRESHOLD = 10

pc = df.copy()

# Ensure JSON columns are decoded, even if df was freshly loaded.
def decode(value, default):
    if isinstance(value, (list, dict)):
        return value
    return json.loads(value) if isinstance(value, str) and value.strip() else default

for col in ("bids", "reviews", "delays", "review_length",
            "review_details", "discussion_initiator"):
    pc[col] = pc[col].map(lambda v: decode(v, []))

for col in ("comments_length", "extra_comments_length"):
    pc[col] = pc[col].map(lambda v: decode(v, {}))

pc["reviewer"] = pc["given_name"].fillna("") + " " + pc["family_name"].fillna("")
pc["email"] = pc["email"].str.strip().str.lower()
pc["n_reviews"] = pc["reviews"].map(len)
pc["n_bids"] = pc["bids"].map(len)

# All reviews must have known, nonpositive delays.
# Reviewers without submitted reviews are ineligible.
pc["on_time"] = pc.apply(
    lambda r: r["n_reviews"] > 0
    and len(r["delays"]) == r["n_reviews"]
    and all(isinstance(d, int) and d <= 0 for d in r["delays"]),
    axis=1
)

# Cumulative FINAL review wordcount.
pc["review_words"] = pc["review_length"].map(
    lambda lengths: sum(lengths)
    if all(v is not None for v in lengths) else float("nan")
)

# Cumulative comment wordcount, including extra comments.
def comment_lengths(row):
    return [
        n
        for col in ("comments_length", "extra_comments_length")
        for lengths in row[col].values()
        for n in lengths
    ]

def total_comment_words(row):
    lengths = comment_lengths(row)
    return (
        sum(lengths)
        if all(n is not None for n in lengths)
        else float("nan")
    )

pc["comment_words"] = pc.apply(total_comment_words, axis=1)

# Count DISTINCT papers, not discussion rounds.
pc["initiated_papers"] = pc["discussion_initiator"].map(
    lambda events: len({
        int(event["paper_id"]) for event in events
    })
)
pc["initiated_rounds"] = pc["discussion_initiator"].map(len)

# SPEEDY: second review submitted OVERALL,
# measured from the beginning of its corresponding round.
def second_review_hours(details):
    if len(details) < 2 or any(
        not r.get("first_submitted_at") for r in details
    ):
        return float("nan")

    ordered = sorted(details, key=lambda r: (
        pd.Timestamp(r["first_submitted_at"]),
        r["paper_id"],
        r.get("review_id") or 0
    ))

    second = ordered[1]

    if not second.get("round_start"):
        return float("nan")

    hours = (
        pd.Timestamp(second["first_submitted_at"])
        - pd.Timestamp(second["round_start"])
    ).total_seconds() / 3600

    return max(0.0, hours)

pc["second_review_hours"] = pc["review_details"].map(
    second_review_hours
)

# Determine whether the venue has configured topics.
topics = pc["topics?"].astype(str).str.strip().str.lower()
topics_required = topics.isin(["true", "false"]).any()
pc["topics_complete"] = topics.eq("true")

# ===================== RECOGNITIONS =====================

awards = {
    "COMMENTER": (
        pc.loc[pc["comment_words"].gt(0)]
        .sort_values(["comment_words", "email"],
                     ascending=[False, True])
        .head(TOP_N)
    ),

    "SPEEDY": (
        pc.loc[
            pc["on_time"] & pc["second_review_hours"].notna()
        ]
        .sort_values(["second_review_hours", "email"])
        .head(TOP_N)
    ),

    "ELABORATE": (
        pc.loc[
            pc["on_time"] & pc["review_words"].notna()
        ]
        .sort_values(["review_words", "email"],
                     ascending=[False, True])
        .head(TOP_N)
    ),

    "COMMITTED": (
        pc.loc[
            pc["on_time"]
            & pc["n_bids"].gt(BID_THRESHOLD)
            & (pc["topics_complete"] if topics_required else True)
        ]
        .sort_values("email")
    ),

    "ALACRITOUS": (
        pc.loc[pc["initiated_papers"].ge(2)]
        .sort_values(
            ["initiated_papers", "initiated_rounds", "email"],
            ascending=[False, False, True]
        )
    )
}

# FIREFIGHTER is based on a manually verified list.
if FIREFIGHTERS:
    manual = {email.strip().lower() for email in FIREFIGHTERS}
    unknown = manual - set(pc["email"])

    if unknown:
        raise ValueError(f"Unknown FIREFIGHTER emails: {sorted(unknown)}")

    awards["FIREFIGHTER"] = (
        pc.loc[pc["email"].isin(manual)].sort_values("email")
    )

# ===================== DISPLAY RESULTS =====================

score_cols = {
    "COMMENTER": "comment_words",
    "SPEEDY": "second_review_hours",
    "ELABORATE": "review_words",
    "COMMITTED": "n_bids",
    "FIREFIGHTER": None,
    "ALACRITOUS": "initiated_papers",
}

ranked = {"COMMENTER", "SPEEDY", "ELABORATE"}
rows = []

for award in (
    "COMMENTER", "SPEEDY", "ELABORATE",
    "COMMITTED", "FIREFIGHTER", "ALACRITOUS"
):
    if award not in awards:
        print("FIREFIGHTER: not evaluated (add verified emails).")
        continue

    for i, person in enumerate(
        awards[award].itertuples(index=False), start=1
    ):
        score = (
            getattr(person, score_cols[award])
            if score_cols[award]
            else "manually verified"
        )

        rows.append({
            "recognition": award,
            "rank": i if award in ranked else None,
            "reviewer": person.reviewer,
            "email": person.email,
            "score": score,
        })

winners_df = pd.DataFrame(
    rows,
    columns=["recognition", "rank", "reviewer", "email", "score"]
)

for award, group in winners_df.groupby("recognition", sort=False):
    print(f"{award} ({len(group)} recipients)")
    display(group.reset_index(drop=True))

# Optional: save results
# winners_df.to_csv("output/recognition_winners_notebook.csv", index=False)


In [ ]:
# TOP REVIEWER: one self-contained Jupyter cell, using the existing `df`.
import json
import math
import pandas as pd
from IPython.display import display

# ---- USER CONFIGURATION --------------------------------------------------
BID_THRESHOLD = 10          # Set an integer, e.g. 20. None = NOT YET EVALUATED.
TOP_PERCENT = 0.30
REQUIRED_POINTS = 6

# Optional manual inputs. None = unknown; an empty set/dict = verified none.
FIREFIGHTER_EMAILS = None     # e.g. {"one@university.edu", "two@company.com"}
UPVOTE_SCORES = None          # e.g. {"one@university.edu": 2, ...}
# UPVOTE_SCORES must cover EVERY reviewer with >=1 submitted review,
# including zeros; choose ONE consistent measure (e.g. total upvotes).
# --------------------------------------------------------------------------


def as_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        return json.loads(value) if value.strip() else []
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    raise ValueError(f"Expected a JSON list, got: {value!r}")


def timestamp(value):
    if not value:
        return None
    try:
        result = pd.Timestamp(value)
        return result.tz_convert("UTC") if result.tzinfo is not None else None
    except (TypeError, ValueError):
        return None


def topic_flag(value):
    value = str(value).strip().casefold()
    return {"true": True, "false": False}.get(value)  # N/A -> unknown


def on_time_flag(delays, review_count):
    if review_count == 0:
        return False
    # A known late submission is sufficient to fail, even if another
    # submission has incomplete timestamps.
    if any(isinstance(d, (int, float)) and not isinstance(d, bool)
           and d > 0 for d in delays):
        return False
    if len(delays) != review_count or any(
        isinstance(d, bool) or not isinstance(d, (int, float))
        or not math.isfinite(d) for d in delays
    ):
        return None
    return True


def early_all_flag(details, review_count):
    """All non-exempt reviews at least 48h before their ORIGINAL deadlines.

    Reviews assigned in the seven days before the deadline (or later)
    are exempt. At least one non-exempt review is required for a point.
    """
    if review_count == 0:
        return False
    if len(details) != review_count:
        return None
    evaluated = 0
    incomplete = False
    failed = False
    for review in details:
        assigned = timestamp(review.get("assigned_at"))
        submitted = timestamp(review.get("first_submitted_at"))
        deadline = timestamp(review.get("deadline"))
        if assigned is None or submitted is None or deadline is None:
            incomplete = True
            continue
        if deadline - assigned <= pd.Timedelta(days=7):
            continue  # late assignment: review exempt for THIS criterion
        evaluated += 1
        if submitted > deadline - pd.Timedelta(hours=48):
            failed = True
    if failed:
        return False
    if incomplete:
        return None
    return evaluated > 0


def top_cutoff(values, fraction=TOP_PERCENT):
    """Smallest score in top ceil(fraction*N); ties at cutoff included."""
    if not values:
        return None
    n = max(1, math.ceil(fraction * len(values)))
    return sorted(values, reverse=True)[n - 1]


# Read already-loaded DataFrame (JSON fields may already be decoded).
people = []
for _, row in df.iterrows():
    email = str(row["email"]).strip().casefold()
    reviews = as_list(row["reviews"])
    details = as_list(row["review_details"])
    delays = as_list(row["delays"])
    lengths = as_list(row["review_length"])
    bids = as_list(row["bids"])
    initiations = as_list(row["discussion_initiator"])
    n_reviews = len(reviews)

    valid_lengths = (
        n_reviews > 0 and len(lengths) == n_reviews
        and all(isinstance(w, (int, float)) and not isinstance(w, bool)
                and math.isfinite(w) and w >= 0 for w in lengths)
    )
    avg_length = (sum(lengths) / n_reviews) if valid_lengths else None
    comments = pd.to_numeric(row["all_comments_totCount"], errors="coerce")

    people.append({
        "reviewer": str(row["given_name"]).strip() + " " + str(row["family_name"]).strip(),
        "email": email,
        "n_bids": len(bids),
        "topics_set": topic_flag(row["topics?"]),
        "n_reviews": n_reviews,
        "n_comments": int(comments) if pd.notna(comments) else None,
        "avg_review_words": avg_length,
        "n_discussions_initiated": len(initiations),
        "no_late_reviews": on_time_flag(delays, n_reviews),
        "all_reviews_48h_early": early_all_flag(details, n_reviews),
    })

if len({r["email"] for r in people}) != len(people):
    raise ValueError("Duplicate PC email addresses in df")
if not (0 < TOP_PERCENT <= 1):
    raise ValueError("TOP_PERCENT must be between 0 and 1")
if BID_THRESHOLD is not None and (
    isinstance(BID_THRESHOLD, bool) or not isinstance(BID_THRESHOLD, int)
    or BID_THRESHOLD < 1
):
    raise ValueError("BID_THRESHOLD must be an integer >= 1, or None")

# Comments: rank against ALL eligible PC members (including zeros).
comment_cutoff = top_cutoff([
    r["n_comments"] for r in people if r["n_comments"] is not None
])

# Mean review length: rank among people who submitted >=1 review.
review_cutoff = top_cutoff([
    r["avg_review_words"] for r in people
    if r["n_reviews"] > 0 and r["avg_review_words"] is not None
])

# Upvotes are optional, but partial reporting would bias the percentile.
upvote_cutoff = None
if UPVOTE_SCORES is not None:
    upvotes = {str(k).strip().casefold(): v for k, v in UPVOTE_SCORES.items()}
    active_emails = {r["email"] for r in people if r["n_reviews"] > 0}
    missing = active_emails - set(upvotes)
    if missing:
        raise ValueError(
            f"UPVOTE_SCORES must cover all reviewers; missing {len(missing)} "
            f"(e.g., {sorted(missing)[0]})"
        )
    if any(not isinstance(upvotes[e], (int, float)) or isinstance(upvotes[e], bool)
           or not math.isfinite(upvotes[e]) or upvotes[e] < 0 for e in active_emails):
        raise ValueError("Upvote scores must be nonnegative finite numbers")
    upvote_cutoff = top_cutoff([upvotes[e] for e in active_emails])

firefighters = (
    None if FIREFIGHTER_EMAILS is None
    else {str(e).strip().casefold() for e in FIREFIGHTER_EMAILS}
)

criterion_names = [
    "C1_bids", "C2_topics", "C3_never_late", "C4_top30_comments",
    "C5_at_least_3_reviews", "C6_top30_avg_review_length",
    "C7_top30_upvotes", "C8_discussion_initiated",
    "C9_all_nonexempt_reviews_48h_early", "C10_extra_review",
]

results = []
for r in people:
    u = upvotes.get(r["email"]) if UPVOTE_SCORES is not None else None
    flags = [
        (r["n_bids"] >= BID_THRESHOLD) if BID_THRESHOLD is not None else None,
        r["topics_set"],
        r["no_late_reviews"],
        (r["n_comments"] > 0 and r["n_comments"] >= comment_cutoff)
        if r["n_comments"] is not None and comment_cutoff is not None else None,
        r["n_reviews"] >= 3,
        (r["avg_review_words"] > 0 and r["avg_review_words"] >= review_cutoff)
        if r["avg_review_words"] is not None and review_cutoff is not None else
        (False if r["n_reviews"] == 0 else None),
        (u > 0 and u >= upvote_cutoff) if u is not None and upvote_cutoff is not None
        else (False if UPVOTE_SCORES is not None and r["n_reviews"] == 0 else None),
        r["n_discussions_initiated"] >= 1,
        r["all_reviews_48h_early"],
        (r["email"] in firefighters) if firefighters is not None else None,
    ]
    known_points = sum(flag is True for flag in flags)
    unknown_count = sum(flag is None for flag in flags)
    possible_points = known_points + unknown_count
    status = (
        "QUALIFIED" if known_points >= REQUIRED_POINTS else
        "POSSIBLE" if possible_points >= REQUIRED_POINTS else
        "CANNOT QUALIFY"
    )
    results.append({
        **r,
        **dict(zip(criterion_names, flags)),
        "known_points": known_points,
        "unknown_criteria": unknown_count,
        "possible_points": possible_points,
        "status": status,
    })

top_reviewers_df = pd.DataFrame(results).sort_values(
    ["known_points", "possible_points", "email"],
    ascending=[False, False, True],
).reset_index(drop=True)

print(f"Top {TOP_PERCENT:.0%} cutoff: {comment_cutoff} comments; "
      f"{review_cutoff:.1f} average review words")
if BID_THRESHOLD is None:
    print("BID_THRESHOLD is unset: bids remain UNKNOWN, not false.")
if UPVOTE_SCORES is None or FIREFIGHTER_EMAILS is None:
    print("Manual upvotes / extra-review criteria remain UNKNOWN until provided.")
print("\nStatus counts:")
display(top_reviewers_df["status"].value_counts().rename_axis("status").to_frame("PC members"))
print("\nConfirmed and possible TOP REVIEWER candidates:")
display(top_reviewers_df.loc[
    top_reviewers_df["status"].ne("CANNOT QUALIFY"),
    ["reviewer", "email", "known_points", "unknown_criteria", "possible_points", "status"]
    + criterion_names
])

# Optionally export the complete, inspectable matrix:
# top_reviewers_df.to_csv("output/top_reviewer_criteria.csv", index=False)


In [ ]:
top_reviewers_df.loc[
    top_reviewers_df["status"].ne("CANNOT QUALIFY"),
    ["reviewer", "email", "known_points", "unknown_criteria", "possible_points", "status"]
    + criterion_names
]

In [ ]:
# 8A. Load firefighter / upvote CSVs and merge them into df.
from pathlib import Path
import pandas as pd

# Paths are relative to the Jupyter notebook's working directory.
FIREFIGHTERS_CSV = Path('../data/aisec26/aisec26-firefighters.csv')  # columns: email, paper, notes
UPVOTES_CSV = Path('../data/aisec26/aisec26-upvotes.csv')              # columns: email, #upvotes

# Signed net upvotes are allowed. Blank scores and unlisted PC members
# receive a score of zero. Invalid nonnumeric entries are rejected.


def normalize_email_column(table, source):
    table = table.copy()
    table['email'] = table['email'].astype('string').str.strip().str.casefold()
    if table['email'].isna().any() or table['email'].eq('').any():
        raise ValueError(f'{source}: missing/blank email addresses')
    return table


def load_required_csv(filename, required_columns):
    if not filename.is_file():
        raise FileNotFoundError(f'Missing {filename}; update the path above.')
    table = pd.read_csv(filename, encoding='utf-8-sig', dtype=str, keep_default_na=False)
    table.columns = table.columns.str.strip()
    missing = set(required_columns) - set(table.columns)
    if missing:
        raise ValueError(f'{filename}: missing required columns: {sorted(missing)}')
    return normalize_email_column(table, filename)


# Each firefighter row represents an extra review of one paper. A reviewer
# can have multiple rows; duplicate (email, paper) pairs count only once.
firefighters_csv = load_required_csv(
    FIREFIGHTERS_CSV, ['email', 'paper', 'notes']
)
firefighters_csv['paper'] = firefighters_csv['paper'].str.strip()
firefighters_csv['notes'] = firefighters_csv['notes'].str.strip()
if firefighters_csv['paper'].eq('').any():
    raise ValueError('firefighters.csv: every row must identify a paper')
number_of_duplicates = firefighters_csv.duplicated(['email', 'paper']).sum()
firefighters_csv = firefighters_csv.drop_duplicates(['email', 'paper'])

firefighter_summary = (
    firefighters_csv.groupby('email', as_index=False)
    .agg(
        extra_review_count=('paper', 'nunique'),
        extra_review_papers=('paper', list),
        extra_review_notes=('notes', list),
    )
)

# Upvotes are per REVIEWER, not per paper; duplicates are ambiguous and rejected.
upvotes_csv = load_required_csv(UPVOTES_CSV, ['email', '#upvotes'])
if upvotes_csv['email'].duplicated().any():
    duplicates = upvotes_csv.loc[upvotes_csv['email'].duplicated(), 'email'].tolist()
    raise ValueError(f'{UPVOTES_CSV}: duplicate email(s): {duplicates[:5]}')
# Blank scores count as zero; signed integers (e.g., -3, 0, +4) are valid.
blank_upvote_rows = upvotes_csv['#upvotes'].str.strip().eq('').sum()
upvotes_csv['upvotes'] = pd.to_numeric(
    upvotes_csv['#upvotes'].str.strip().replace('', '0'), errors='coerce'
)
if upvotes_csv['upvotes'].isna().any() or (upvotes_csv['upvotes'] % 1 != 0).any():
    raise ValueError(f'{UPVOTES_CSV}: #upvotes must be a signed integer or blank')
upvotes_csv['upvotes'] = upvotes_csv['upvotes'].astype('int64')

# Rerunning this cell is safe: replace only the previous manual-data columns.
manual_columns = [
    'extra_review_count', 'extra_review_papers', 'extra_review_notes',
    'firefighter', 'upvotes', 'upvotes_missing_from_csv',
]
df = df.drop(columns=manual_columns, errors='ignore').copy()
df = normalize_email_column(df, 'pc_metrics.csv')
if df['email'].duplicated().any():
    raise ValueError('pc_metrics.csv: PC email addresses must be unique')

known_emails = set(df['email'])
for filename, table in ((FIREFIGHTERS_CSV, firefighters_csv),
                        (UPVOTES_CSV, upvotes_csv)):
    unknown = set(table['email']) - known_emails
    if unknown:
        raise ValueError(f'{filename}: unrecognized PC email(s): {sorted(unknown)[:5]}')

df = df.merge(firefighter_summary, on='email', how='left', validate='one_to_one')
df = df.merge(upvotes_csv[['email', 'upvotes']], on='email',
              how='left', validate='one_to_one')

df['extra_review_count'] = df['extra_review_count'].fillna(0).astype(int)
for col in ('extra_review_papers', 'extra_review_notes'):
    df[col] = df[col].map(lambda x: x if isinstance(x, list) else [])
df['firefighter'] = df['extra_review_count'].ge(1)
df['upvotes_missing_from_csv'] = df['upvotes'].isna()  # audit information
missing_vote_rows = int(df['upvotes_missing_from_csv'].sum())
df['upvotes'] = df['upvotes'].fillna(0).astype('int64')

print(f'Loaded {len(firefighters_csv)} unique emergency-review records '
      f'for {df["firefighter"].sum()} PC members '
      f'({number_of_duplicates} duplicate paper records ignored).')
print(f'Loaded upvote scores for {len(upvotes_csv)} PC members; '
      f'{int(blank_upvote_rows)} blank scores and {missing_vote_rows} '
      'unlisted PC members treated as 0. Negative scores are preserved.')

display(df[['reviewer', 'email', 'extra_review_count',
            'extra_review_papers', 'upvotes']].head())


In [ ]:
df[df['firefighter']==True]

In [ ]:
# 8B. Recompute TOP REVIEWER from the df enriched by Cell 8A.
import json
import math
import pandas as pd
from IPython.display import display

# ---- USER CONFIGURATION --------------------------------------------------
BID_THRESHOLD = 10    # Set your threshold, e.g. 20. None = unknown criterion.
TOP_PERCENT = 0.30
REQUIRED_POINTS = 7     # 6 out of 10 total criteria
# --------------------------------------------------------------------------

if not {'extra_review_count', 'upvotes'}.issubset(df.columns):
    raise RuntimeError('Run Cell 8A first to load firefighters and upvotes.')


def as_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        return json.loads(value) if value.strip() else []
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    raise ValueError(f'Expected a JSON list, got: {value!r}')


def timestamp(value):
    if not value:
        return None
    try:
        result = pd.Timestamp(value)
        return result.tz_convert('UTC') if result.tzinfo is not None else None
    except (TypeError, ValueError):
        return None


def topic_flag(value):
    value = str(value).strip().casefold()
    return {'true': True, 'false': False}.get(value)  # N/A -> unknown


def on_time_flag(delays, review_count):
    if review_count == 0:
        return False
    # A known late submission fails this criterion even if another is unknown.
    if any(isinstance(d, (int, float)) and not isinstance(d, bool)
           and d > 0 for d in delays):
        return False
    if len(delays) != review_count or any(
        isinstance(d, bool) or not isinstance(d, (int, float))
        or not math.isfinite(d) for d in delays
    ):
        return None
    return True


def early_all_flag(details, review_count):
    """Require each non-exempt review at least 48h before its original deadline.

    A review assigned within 7 days of its deadline (including AFTER the
    deadline) is exempt from THIS criterion, not awarded early-submission credit.
    At least one non-exempt review must exist to earn the point.
    """
    if review_count == 0:
        return False
    if len(details) != review_count:
        return None
    evaluated = 0
    incomplete = False
    failed = False
    for review in details:
        assigned = timestamp(review.get('assigned_at'))
        submitted = timestamp(review.get('first_submitted_at'))
        deadline = timestamp(review.get('deadline'))
        if assigned is None or submitted is None or deadline is None:
            incomplete = True
            continue
        if deadline - assigned <= pd.Timedelta(days=7):
            continue
        evaluated += 1
        if submitted > deadline - pd.Timedelta(hours=48):
            failed = True
    if failed:
        return False
    if incomplete:
        return None
    return evaluated > 0


def top_cutoff(values, fraction):
    """Smallest value in the top ceil(fraction*N), with ties included."""
    if not values:
        return None
    count = max(1, math.ceil(fraction * len(values)))
    return sorted(values, reverse=True)[count - 1]


if not (0 < TOP_PERCENT <= 1):
    raise ValueError('TOP_PERCENT must be in (0, 1].')
if BID_THRESHOLD is not None and (
    isinstance(BID_THRESHOLD, bool) or not isinstance(BID_THRESHOLD, int)
    or BID_THRESHOLD < 1
):
    raise ValueError('BID_THRESHOLD must be an integer >= 1, or None.')

people = []
for _, row in df.iterrows():
    reviews = as_list(row['reviews'])
    n_reviews = len(reviews)
    details = as_list(row['review_details'])
    delays = as_list(row['delays'])
    lengths = as_list(row['review_length'])
    bids = as_list(row['bids'])
    initiations = as_list(row['discussion_initiator'])
    valid_lengths = (
        n_reviews > 0 and len(lengths) == n_reviews
        and all(isinstance(w, (int, float)) and not isinstance(w, bool)
                and math.isfinite(w) and w >= 0 for w in lengths)
    )
    avg_length = sum(lengths) / n_reviews if valid_lengths else None
    comments = pd.to_numeric(row['all_comments_totCount'], errors='coerce')
    votes = row['upvotes']

    people.append({
        'reviewer': row['reviewer'],
        'email': row['email'],
        'n_bids': len(bids),
        'topics_set': topic_flag(row['topics?']),
        'n_reviews': n_reviews,
        'n_comments': int(comments) if pd.notna(comments) else None,
        'avg_review_words': avg_length,
        'upvotes': int(votes) if pd.notna(votes) else None,
        'extra_review_count': int(row['extra_review_count']),
        'n_discussions_initiated': len(initiations),
        'no_late_reviews': on_time_flag(delays, n_reviews),
        'all_reviews_48h_early': early_all_flag(details, n_reviews),
    })

if len({r['email'] for r in people}) != len(people):
    raise ValueError('Duplicate PC email addresses in df.')

# Comments are ranked across ALL PC members; ties at cutoff count.
comment_cutoff = top_cutoff(
    [r['n_comments'] for r in people if r['n_comments'] is not None],
    TOP_PERCENT,
)
# Average review length: only reviewers with at least one review.
review_cutoff = top_cutoff(
    [r['avg_review_words'] for r in people if r['avg_review_words'] is not None],
    TOP_PERCENT,
)
# To avoid biased percentiles, upvotes are ranked only with COMPLETE coverage
# of all PC members who submitted at least one review, including zero scores.
active = [r for r in people if r['n_reviews'] > 0]
missing_active = [r['email'] for r in active if r['upvotes'] is None]
upvote_cutoff = (
    top_cutoff([r['upvotes'] for r in active], TOP_PERCENT)
    if active and not missing_active else None
)

criterion_names = [
    'C1_bids', 'C2_topics', 'C3_never_late', 'C4_top30_comments',
    'C5_at_least_3_reviews', 'C6_top30_avg_review_length',
    'C7_top30_upvotes', 'C8_discussion_initiated',
    'C9_all_nonexempt_reviews_48h_early', 'C10_extra_review',
]

results = []
for r in people:
    flags = [
        r['n_bids'] >= BID_THRESHOLD if BID_THRESHOLD is not None else None,
        r['topics_set'],
        r['no_late_reviews'],
        (r['n_comments'] > 0 and r['n_comments'] >= comment_cutoff)
        if r['n_comments'] is not None and comment_cutoff is not None else None,
        r['n_reviews'] >= 3,
        (r['avg_review_words'] > 0 and r['avg_review_words'] >= review_cutoff)
        if r['avg_review_words'] is not None and review_cutoff is not None
        else (False if r['n_reviews'] == 0 else None),
        (r['upvotes'] > 0 and r['upvotes'] >= upvote_cutoff)
        if r['n_reviews'] > 0 and upvote_cutoff is not None
        else (False if r['n_reviews'] == 0 else None),
        r['n_discussions_initiated'] >= 1,
        r['all_reviews_48h_early'],
        r['extra_review_count'] >= 1,
    ]
    known_points = sum(flag is True for flag in flags)
    unknown_count = sum(flag is None for flag in flags)
    possible_points = known_points + unknown_count
    status = (
        'QUALIFIED' if known_points >= REQUIRED_POINTS else
        'POSSIBLE' if possible_points >= REQUIRED_POINTS else
        'CANNOT QUALIFY'
    )
    results.append({
        **r,
        **dict(zip(criterion_names, flags)),
        'known_points': known_points,
        'unknown_criteria': unknown_count,
        'possible_points': possible_points,
        'status': status,
    })

top_reviewers_df = pd.DataFrame(results).sort_values(
    ['known_points', 'possible_points', 'email'],
    ascending=[False, False, True],
).reset_index(drop=True)

print(f'Top {TOP_PERCENT:.0%} cutoffs: '
      f'{comment_cutoff} comments; '
      f'{review_cutoff:.1f} mean review words; '
      f'{upvote_cutoff if upvote_cutoff is not None else "UNKNOWN"} upvotes')
if BID_THRESHOLD is None:
    print('BID_THRESHOLD is unset; bids criterion remains UNKNOWN.')
if missing_active:
    print(f'Upvote percentile is unavailable: {len(missing_active)} active '
          'reviewers lack upvote scores (see Cell 8A).')
print('\nStatus counts:')
display(top_reviewers_df['status'].value_counts().rename_axis('status').to_frame('PC members'))
print('\nConfirmed and possible TOP REVIEWER candidates:')
display(top_reviewers_df.loc[
    top_reviewers_df['status'].ne('CANNOT QUALIFY'),
    ['reviewer', 'email', 'known_points', 'unknown_criteria', 'possible_points',
     'status', 'upvotes', 'extra_review_count'] + criterion_names,
])

# Optional: export the full 10-criterion evidence matrix.
# top_reviewers_df.to_csv('output/top_reviewer_criteria.csv', index=False)


In [ ]:
import json
import pandas as pd

def parse_json(value, default):
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, str) and value.strip():
        return json.loads(value)
    return default

def count_comments(value):
    data = parse_json(value, {})
    return sum(int(n) for n in data.values())

def count_comment_words(value):
    data = parse_json(value, {})
    return sum(
        int(n)
        for lengths in data.values()
        for n in lengths
        if n is not None
    )

stats = {
    "Total comments": (
        df["comments"].map(count_comments).sum()
        + df["extra_comments"].map(count_comments).sum()
    ),
    "Total reviews": df["reviews"].map(
        lambda x: len(parse_json(x, []))
    ).sum(),
    "Total comment words": (
        df["comments_length"].map(count_comment_words).sum()
        + df["extra_comments_length"].map(count_comment_words).sum()
    ),
    "Total review words": df["review_length"].map(
        lambda x: sum(
            int(n) for n in parse_json(x, [])
            if n is not None
        )
    ).sum()
}

display(pd.DataFrame(
    [{"Metric": k, "Value": f"{int(v):,}"} for k, v in stats.items()]
))